In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 09 — Reference coding solutions

These are instructor reference implementations, not evidence of your independent mastery. Run after attempting your own version. All examples use explicit small inputs and the declared dataset partitions.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
from itertools import combinations
import numpy as np
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/manifest.json').exists() and (p/'src/shape_lab').exists()),None)
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted course folder.')
sys.path.insert(0,str(ROOT/'src'))
from shape_lab.data import load_iris_data,load_digit_data,digit_example
from shape_lab.algebra import closure,validate_complex,rank_mod2,boundary_matrix,betti_numbers
from shape_lab.topology import component_labels
from shape_lab.persistence import ordered_filtration,rips_filtration,reduce_boundary,persistent_homology,diagram,pixel_filtration,finite_bottleneck
print('Working inside',ROOT.name,'— no network data downloads.')

Working inside course — no network data downloads.


## 09.C1 — Order a filtration correctly

Given valid (simplex,time) entries, sort by time, then vertex count, then tuple.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Python compares tuples lexicographically, field by field.

</details>

In [3]:
def filtration_order(entries):
    return sorted(entries,key=lambda item:(item[1],len(item[0]),item[0]))

In [4]:
f=filtration_order([((0,1),0),((1,),0),((0,),0)])
assert [s for s,v in f]==[(0,),(1,),(0,1)]
print("Faces precede cofaces in a tied filtration")

Faces precede cofaces in a tied filtration


**Why this works and what it does not establish:** Tie ordering is an implementation detail, but it must respect the filtration. Positive-length intervals should not depend on an arbitrary valid ordering of simultaneous unrelated simplices.

## 09.C2 — Construct sparse boundary columns

Given an already valid ordered filtration, return one set of row indices per boundary column over F2. Vertices have empty boundary.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

The row index comes from the filtration order, not the numerical vertex ID.

</details>

In [5]:
def sparse_boundary_columns(f):
    index={s:i for i,(s,v) in enumerate(f)}
    return [{index[face] for face in combinations(s,len(s)-1)} if len(s)>1 else set() for s,v in f]

In [6]:
f=ordered_filtration([((0,),0),((1,),0),((0,1),1)])
assert sparse_boundary_columns(f)==[set(),set(),{0,1}]
print("One edge boundary contains its two vertex rows")

One edge boundary contains its two vertex rows


**Why this works and what it does not establish:** The sparse set stores which boundary coefficients are one. Indexing by simplex identity is necessary once rows represent edges and triangles rather than just vertices.

## 09.C3 — Reduce sparse columns

Copy the columns. Repeatedly XOR a previous column with the same largest row until that pivot is unique. Return reduced columns and pivot-row-to-column mapping.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

The largest nonzero row is often called low. XOR cancels it when the pivots match.

</details>

In [7]:
def my_reduce(columns):
    reduced=[];owners={}
    for j,original in enumerate(columns):
        col=set(original)
        while col and max(col) in owners:
            col ^= reduced[owners[max(col)]]
        if col: owners[max(col)]=j
        reduced.append(col)
    return reduced,owners

In [8]:
columns=[set(),set(),set(),{0,1},{1,2},{0,2}]
reduced,owners=my_reduce(columns)
assert reduced[-1]==set() and len(owners)==2
assert columns[-1]=={0,2}
print("The closing edge reduces to zero without mutating input")

The closing edge reduces to zero without mutating input


**Why this works and what it does not establish:** The zero column indicates a homology birth in the filtered complex. Later columns can pair with that birth; zero does not imply that the class survives forever.

## 09.C4 — Extract paired birth and death times

Given a filtration and pivot mapping from reduced boundary columns, return (dimension,birth,death) for each finite positive-length pair.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

The dimension belongs to the birth simplex, one below its killing coface.

</details>

In [9]:
def finite_pair_times(f,pivots):
    return sorted((len(f[i][0])-1,f[i][1],f[j][1]) for i,j in pivots.items() if f[j][1]>f[i][1])

In [10]:
X=np.array([[0,0],[1,0],[1,1],[0,1]],float)
f,reduced,pairs,operations=reduce_boundary(rips_filtration(X))
result=finite_pair_times(f,pairs)
assert len([r for r in result if r[0]==1])==1
assert np.allclose([r[1:] for r in result if r[0]==1],[[1,np.sqrt(2)]])
print("Positive finite pairs:",result)

Positive finite pairs: [(0, 0.0, 1.0), (0, 0.0, 1.0), (0, 0.0, 1.0), (1, 1.0, 1.4142135623730951)]


**Why this works and what it does not establish:** Unpaired zero columns require a separate pass for surviving classes. This exercise deliberately returns only finite pairs and labels that restriction.

## 09.C5 — Compute H0 merges independently

On a nonempty small point cloud, sort all pairwise distances and use union-find to collect merge distances. Include zero-distance merges.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

An edge merges components only when its endpoints currently have different representatives.

</details>

In [11]:
def h0_merge_distances(X):
    X=np.asarray(X,dtype=float);parent=list(range(len(X)))
    def find(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]];i=parent[i]
        return i
    edges=sorted((float(np.linalg.norm(X[i]-X[j])),i,j) for i,j in combinations(range(len(X)),2))
    deaths=[]
    for d,i,j in edges:
        a,b=find(i),find(j)
        if a!=b: parent[b]=a;deaths.append(d)
    return deaths

In [12]:
X=load_iris_data().iloc[:12,1:5].to_numpy()
deaths=h0_merge_distances(X)
D=diagram(persistent_homology(rips_filtration(X,max_homology=0)),0,finite_only=True)
assert len(deaths)==len(X)-1
assert np.allclose([d for d in deaths if d>0],sorted(D[:,1]))
print("Real-point-cloud union-find merge agreement:",len(deaths),"merges")

Real-point-cloud union-find merge agreement: 11 merges


**Why this works and what it does not establish:** This checks H0 with a different algorithmic route. It does not validate H1 by itself and does not independently certify every piece of shared distance code.